In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("../data/test.csv", index_col=0)

# Create copy
df_encoded = df.copy()

# -----------------------
# DATE FEATURES
# -----------------------
df_encoded["Date"] = pd.to_datetime(df_encoded["Date"])

# Long-term trend
df_encoded["time_idx"] = (
    df_encoded["Date"] - df_encoded["Date"].min()
).dt.days

# Intermediate variables
df_encoded["month"] = df_encoded["Date"].dt.month
df_encoded["dayofweek"] = df_encoded["Date"].dt.dayofweek

# Weekend indicator
df_encoded["is_weekend"] = (
    df_encoded["dayofweek"] >= 5
).astype(int)

# Cyclic features
df_encoded["month_sin"] = np.sin(
    2 * np.pi * df_encoded["month"] / 12
)
df_encoded["month_cos"] = np.cos(
    2 * np.pi * df_encoded["month"] / 12
)

df_encoded["dow_sin"] = np.sin(
    2 * np.pi * df_encoded["dayofweek"] / 7
)
df_encoded["dow_cos"] = np.cos(
    2 * np.pi * df_encoded["dayofweek"] / 7
)

# Remove intermediate columns and original Date
df_encoded.drop(
    columns=["Date", "month", "dayofweek"],
    inplace=True
)

# -----------------------
# STORE-PRODUCT FEATURES
# -----------------------
df_encoded["Store_Product"] = (
    df_encoded["Store ID"].astype(str)
    + "_"
    + df_encoded["Product ID"].astype(str)
)

# Frequency encoding
df_encoded["store_freq"] = (
    df_encoded["Store ID"]
    .map(df_encoded["Store ID"].value_counts())
)

df_encoded["product_freq"] = (
    df_encoded["Product ID"]
    .map(df_encoded["Product ID"].value_counts())
)

df_encoded["store_product_freq"] = (
    df_encoded["Store_Product"]
    .map(df_encoded["Store_Product"].value_counts())
)

# -----------------------
# CATEGORICAL FEATURES
# -----------------------
CARDINALITY_THRESHOLD = 10

categorical_cols = df_encoded.select_dtypes(include=["object"]).columns

low_card_cols = []
high_card_cols = []

# Exclude Store ID, Product ID, Store_Product
special_cols = ["Store ID", "Product ID", "Store_Product"]

for col in categorical_cols:
    if col in special_cols:
        continue

    if df_encoded[col].nunique() <= CARDINALITY_THRESHOLD:
        low_card_cols.append(col)
    else:
        high_card_cols.append(col)

print("Low cardinality:", low_card_cols)
print("High cardinality:", high_card_cols)

# -----------------------
# FREQUENCY ENCODE HIGH CARDINALITY
# -----------------------
for col in high_card_cols:
    freq_map = df_encoded[col].value_counts()

    df_encoded[col + "_freq"] = (
        df_encoded[col].map(freq_map)
    )

    df_encoded.drop(columns=[col], inplace=True)

# -----------------------
# DROP RAW STORE/PRODUCT COLUMNS
# -----------------------
df_encoded.drop(
    columns=["Store ID", "Product ID", "Store_Product"],
    inplace=True
)

# -----------------------
# ONE-HOT ENCODE LOW CARDINALITY
# -----------------------
df_encoded = pd.get_dummies(
    df_encoded,
    columns=low_card_cols,
    dtype=int
)

print("\n--- TRANSFORMATION COMPLETE ---")
print("Original shape:", df.shape)
print("Encoded shape:", df_encoded.shape)

display(df_encoded.head(3))

df_encoded.to_csv("transformed_test.csv")

Low cardinality: ['Category', 'Region', 'Weather Condition', 'Seasonality']
High cardinality: []

--- TRANSFORMATION COMPLETE ---
Original shape: (11400, 16)
Encoded shape: (11400, 33)


/tmp/ipykernel_55966/372584786.py:80: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_encoded.select_dtypes(include=["object"]).columns


,Inventory Level,Units Sold,Units Ordered,Price,Discount,Promotion,Competitor Pricing,Epidemic,Demand,time_idx,...,Region_East,Region_North,Region_South,Region_West,Weather Condition_Cloudy,Weather Condition_Rainy,Weather Condition_Snowy,Weather Condition_Sunny,Seasonality_Autumn,Seasonality_Winter
64600,220,42,0,50.11,20,1,48.56,1,44,0,...,0,1,0,0,1,0,0,0,1,0
64601,194,140,126,72.45,5,0,68.39,1,128,0,...,0,1,0,0,1,0,0,0,1,0
64602,577,64,0,48.14,10,1,53.83,1,87,0,...,0,1,0,0,1,0,0,0,1,0
